# B2 – Odnosi med spremenljivkami

**Razdelek B · samostojno delo · Telekom**

Pri vsaki nalogi povej: **kaj primerjam, koliko zapisov poznam, kateri imenovalec
uporabljam in ali opisujem povezavo ali trdim vzročnost?**


## Priprava

Notebook je samostojen in deluje lokalno ter v Google Colabu. Celice izvajaj od
zgoraj navzdol s **Shift + Enter**. Ob menjavi stolpcev ponovi ustrezne naslednje celice.

Prva celica poišče `utils/orodja.py`; v Colabu po potrebi prenese repozitorij in namesti
knjižnice iz `requirements.txt`. Lokalno jih enkrat namestimo po navodilih v README.
Druga celica pokliče skupni `poskrbi_za_podatke()` in vrne mapo `DATA`.
Funkcija uporabi pripravljene ali že prenesene podatke; manjkajoče prenese s Kaggla.
CSV-je nato beremo iz `DATA`, ne glede na mapo, v kateri je zvezek odprt.
Izvornih vrednosti ne popravljamo in rezultatov vaj ne zapisujemo samodejno v datoteke.


In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


### Skupni podatkovni vhod
`poskrbi_za_podatke()` iz `utils/orodja.py` pripravi skupno mapo podatkov in jo vrne kot `DATA`.

V tem sklopu beremo `DATA / "telekom_izvirni.csv"`: vseh 100 stolpcev z izvirnimi imeni.
Uvod in vizualizacija uporabljata 22-stolpčni slovenski izbor `telekom.csv`;
obe tabeli izhajata iz istih strank in ju pripravi isti podatkovni vhod.


In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_colwidth", 35)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})


In [ ]:
df = pd.read_csv(DATA / "telekom_izvirni.csv")
print(f"Vrstic: {df.shape[0]:,}; stolpcev: {df.shape[1]}")
display(df.head())


### Predlagani stolpci

| Metoda | Začetni primer | Druge možnosti |
|---|---|---|
| Pearson, Spearman, razsevni diagram | `totmrc_Mean` / `rev_Mean` | `mou_Mean` / `rev_Mean`; `totcalls` / `adjqty` |
| Vzorec, ki ga koeficient skrije | `months` / `churn` | `eqpdays` po smiselnih razredih |
| Korelacija s ciljem | Cilj `churn`; izbrane meritve | `eqpdays`, `hnd_price`, `mou_Mean`, `change_mou` |
| Primerjava skupin | `hnd_price` / `churn` | `refurb_new` / `churn`; `area` / `rev_Mean` |
| Kontingenčna tabela | `asl_flag` / `churn` | `new_cell` / `churn`; `refurb_new` / `churn` |
| Matrika | `totcalls`, `adjqty`, `totrev`, `adjrev` | Mere uporabe, prihodka in trajanja razmerja |

`churn` je oznaka 0/1; njegovo povprečje je delež enic. `Customer_ID` je identifikator,
ne meritev. Kod kategorij ne pretvarjamo v naključne številke za korelacijo. Pomen
`asl_flag`, `refurb_new` in drugih kod pri poslovni razlagi preverimo v slovarju.


## 1. Izberi orodje


| Tip para | Orodje | Kaj vedno izpišemo? |
|---|---|---|
| Numerična + numerična | Pearson, Spearman, razsevni diagram | Število znanih parov |
| Kategorija + numerična | `groupby`, povzetki po skupinah | Velikost skupine in število znanih meritev |
| Kategorija + kategorija | `pd.crosstab` | Števila in imenovalec deležev |
| Več numeričnih stolpcev | Korelacijska matrika | Izbor meritev in število znanih parov |

Pri vsaki nalogi izpustimo le zapise brez vrednosti, potrebnih za **to vprašanje**.
Celotne tabele ne čistimo s splošnim `dropna()`.

**Pred izvajanjem:** izberi par iz tabele predlogov in utemelji izbiro metode.


## 2. Pearson, Spearman in razsevni diagram

**Začni:** `totmrc_Mean` / `rev_Mean`. **Ponovi:** `mou_Mean` / `rev_Mean`.
**Dodaten primer:** `totcalls` / `adjqty`.

Pearson meri linearno, Spearman monotono povezanost. Koeficienta uporabljata vse
znane pare. Graf uporablja vzorec; desni prikaz omeji osi na 1.–99. percentil.
To ni brisanje ekstremov ali nov izračun korelacije.


In [ ]:
x = 'totmrc_Mean'
y = 'rev_Mean'
pari = df[[x, y]].dropna()
display(pd.Series({
    "znanih parov": len(pari),
    "Pearson": pari.corr(method="pearson").loc[x, y],
    "Spearman": pari.corr(method="spearman").loc[x, y],
}).to_frame("vrednost").round(3))

# Koeficienta uporabljata vse znane pare; graf reproducibilen vzorec.
vzorec = pari.sample(n=min(2500, len(pari)), random_state=42)
fig, osi = plt.subplots(1, 2, figsize=(11, 4))
for os in osi:
    os.scatter(vzorec[x], vzorec[y], s=9, alpha=0.25)
    os.set(xlabel=x, ylabel=y)
osi[0].set_title("Celoten razpon vzorca")
osi[1].set_title("Povečan prikaz: osi med 1. in 99. percentilom")
osi[1].set_xlim(pari[x].quantile(0.01), pari[x].quantile(0.99))
osi[1].set_ylim(pari[y].quantile(0.01), pari[y].quantile(0.99))
fig.tight_layout()
plt.show()


**Zapiši:** ali se koeficienta razlikujeta? Kaj na grafu opaziš o obliki in ekstremih?

Razlika med koeficientoma sama še ne pove, kaj jo je povzročilo.


## 3. Majhna korelacija lahko skrije vzorec

Primerjaj `months` in `churn`. Poleg koeficientov izračunaj delež odhodov za vsak
mesec ter za širše skupine. Povprečje 0/1 je delež enic; ob njem izpišemo velikost skupine.

Meje so izbrane za raziskovanje tega primera. V zbirki ni podatka o vezavi, zato
morebitnega vrha ne razlagaj kot dokaz izteka enoletne pogodbe.


In [ ]:
pari_meseci = df[["months", "churn"]].dropna().copy()
display(pd.Series({
    "znanih parov": len(pari_meseci),
    "Pearson": pari_meseci.corr(method="pearson").loc["months", "churn"],
    "Spearman": pari_meseci.corr(method="spearman").loc["months", "churn"],
}).to_frame("vrednost").round(3))
po_mesecih = pari_meseci.groupby("months")["churn"].agg(stranke="size", delez_odhodov="mean")
display(po_mesecih.round(3))
pari_meseci["razred_mesecev"] = pd.cut(
    pari_meseci["months"], bins=[0, 10, 12, np.inf], labels=["do 10", "11–12", "13 in več"]
)
display(pari_meseci.groupby("razred_mesecev", observed=True)["churn"].agg(
    stranke="size", delez_odhodov="mean"
).round(3))
fig, os = plt.subplots()
(po_mesecih["delez_odhodov"] * 100).plot(marker="o", markersize=3, ax=os)
os.set(xlabel="Trajanje razmerja (months)", ylabel="Delež odhodov (%)",
       title="Povezavo preverimo tudi po posameznih mesecih", ylim=(0, 100))
fig.tight_layout()
plt.show()


**Preveri:** kaj en sam koeficient skrije? Ali so zadnji meseci enako dobro zastopani?
Zakaj iz grafa še ne moreš razložiti vzroka odhoda?


## 4. Korelacija s ciljem: vrstni red značilk

Cilj je `churn`. Razvrščamo po absolutni korelaciji, pri izpisu ohranimo predznak.
Šibka korelacija ne pomeni nujno neuporabne značilke: ne zajame nelinearnosti ali
kombinacij vhodov. Uporabimo izrecen seznam meritev, brez ID-jev in neurejenih kod.


In [ ]:
cilj = 'churn'
kandidati = ['eqpdays', 'hnd_price', 'mou_Mean', 'rev_Mean', 'totmrc_Mean', 'change_mou', 'change_rev', 'months', 'custcare_Mean', 'phones']
korelacije = df[kandidati + [cilj]].corr(method="spearman", min_periods=30)[cilj].drop(cilj)
korelacije = korelacije.sort_values(key=lambda s: s.abs(), ascending=False)
n = pd.Series({s: len(df[[s, cilj]].dropna()) for s in kandidati}, name="znanih_parov")
display(pd.concat([korelacije.rename("Spearman"), n], axis=1).loc[korelacije.index].round(3))

fig, os = plt.subplots()
korelacije.iloc[::-1].plot.barh(ax=os)
os.axvline(0, color="black", linewidth=0.8)
os.set(xlabel=f"Spearmanov koeficient s ciljem: {cilj}", title="Prvi pregled posameznih značilk")
fig.tight_layout()
plt.show()


**Zapiši:** katera značilka ima največjo absolutno korelacijo? Katera ima negativen
predznak? Zakaj seznam sam še ni model ali dokaz uspešnega ukrepa proti odhodu?


## 5. Primerjava skupin

**Začni:** `hnd_price` / `churn`. **Ponovi:** `refurb_new` / `churn` ali
`area` / `rev_Mean`. Povprečje `churn` je delež odhodov, povprečje prihodka pa
znesek v izvornih enotah. `size` šteje vse vrstice, `count` znane meritve.

Graf prikaže do 15 največjih skupin z vsaj 300 znanimi meritvami; celotna tabela
ostane vidna. Decimalni zapis cen za graf le formatiramo.


In [ ]:
skupina = 'hnd_price'
meritev = 'churn'
prag = 300
po_skupinah = df.groupby(skupina, dropna=False)[meritev].agg(
    vrstic="size", znanih="count", povprecje="mean", mediana="median"
)
display(po_skupinah.round(3))

# Omejimo samo graf; celotna tabela je izpisana zgoraj.
za_graf = po_skupinah.loc[po_skupinah["znanih"].ge(prag)].nlargest(15, "znanih").copy()
if skupina == "hnd_price":
    za_graf.index = za_graf.index.map(lambda v: f"{v:.2f}")
fig, os = plt.subplots(figsize=(10, 4))
za_graf["povprecje"].plot.bar(ax=os)
os.set(xlabel=skupina, ylabel=f"Povprečje: {meritev}",
       title=f"Do 15 največjih skupin z vsaj {prag} znanimi meritvami")
fig.tight_layout()
plt.show()


**Preveri:** ali so skupine podobno velike? Kaj bi bilo narobe s poudarjanjem
največjega deleža v zelo majhni skupini? Kako se spremeni pomen povprečja pri prihodku?


## 6. Kontingenčna tabela in dva imenovalca

**Začni:** `asl_flag` / `churn`. **Ponovi:** `new_cell` ali `refurb_new` proti `churn`.

- Števila povedo, koliko strank je v kombinaciji.
- Deleži po vrsticah opišejo odhode **znotraj kategorije**.
- Deleži po stolpcih opišejo sestavo kategorij **med odšlimi oziroma ostalimi**.

V vseh tabelah uporabimo iste zapise z znanima obema oznakama.


In [ ]:
kategorija = 'asl_flag'
cilj_kategorije = 'churn'
pari_k = df[[kategorija, cilj_kategorije]].dropna()
print(f"Znanih parov: {len(pari_k):,}; izpuščenih: {len(df) - len(pari_k):,}")
stevila = pd.crosstab(pari_k[kategorija], pari_k[cilj_kategorije])
po_vrsticah = pd.crosstab(pari_k[kategorija], pari_k[cilj_kategorije], normalize="index")
po_stolpcih = pd.crosstab(pari_k[kategorija], pari_k[cilj_kategorije], normalize="columns")
print("Število zapisov:")
display(stevila)
print("Deleži po vrsticah (%):")
display((po_vrsticah * 100).round(2))
print("Deleži po stolpcih (%):")
display((po_stolpcih * 100).round(2))

fig, os = plt.subplots()
(po_vrsticah * 100).plot.bar(stacked=True, ax=os)
os.set(xlabel=kategorija, ylabel="Delež zapisov znotraj kategorije (%)",
       title="Porazdelitev znotraj posamezne kategorije", ylim=(0, 100))
os.legend(title=cilj_kategorije, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()


**Zapiši dve povedi:** »Med strankami s kategorijo … je odšlo … %.« in »Med odšlimi
strankami je kategorijo … imelo … %.« Kateri tabeli si uporabil? Zakaj vsota v
napačni smeri ni nujno 100 %?


## 7. Korelacijska matrika in skoraj podvojena informacija

Začni z osmimi izbranimi stolpci, nato enega zamenjaj s `totmrc_Mean` ali `change_mou`.
Poišči pare z absolutno korelacijo nad 0,95. Upoštevaj število znanih parov.

Visoka korelacija in zaokrožen izpis `1,00` še ne pomenita enakih vrednosti v vsaki
vrstici. Razmisli o razlagi koeficientov modela z več skoraj enakimi vhodi.


In [ ]:
stolpci_matrike = ['totcalls', 'adjqty', 'totrev', 'adjrev', 'rev_Mean', 'mou_Mean', 'months', 'eqpdays']
matrika = df[stolpci_matrike].corr(method="pearson", min_periods=30)
znane = df[stolpci_matrike].notna().astype("int64")
n_parov = znane.T.dot(znane)
display(matrika.round(3))
print("Število znanih parov za vsako kombinacijo:")
display(n_parov)

fig, os = plt.subplots(figsize=(10, 7))
slika = os.imshow(matrika, cmap="RdBu_r", vmin=-1, vmax=1)
os.set_xticks(range(len(stolpci_matrike)), labels=stolpci_matrike, rotation=45, ha="right")
os.set_yticks(range(len(stolpci_matrike)), labels=stolpci_matrike)
for vrstica in range(len(stolpci_matrike)):
    for stolpec in range(len(stolpci_matrike)):
        r = matrika.iloc[vrstica, stolpec]
        os.text(stolpec, vrstica, f"{r:.2f}", ha="center", va="center",
                color="white" if abs(r) > 0.65 else "black")
fig.colorbar(slika, ax=os, label="Pearsonov r")
os.set_title("Korelacijska matrika izbranih numeričnih stolpcev")
fig.tight_layout()
plt.show()

# Zgornji trikotnik izpusti diagonalo in dvojno štetje parov.
maska = np.triu(np.ones(matrika.shape, dtype=bool), k=1)
pari_matrike = matrika.where(maska).stack()
mocni_pari = pari_matrike.loc[pari_matrike.abs().gt(0.95)]
display(mocni_pari.rename("Pearsonov r").to_frame())


## 8. Povezanost ni vzročnost: stiki s podporo

Primerjaj delež odhodov ob ničelni in pozitivni meri `custcare_Mean`, nato še znotraj
kvartilov `mou_Mean`. Uporaba lahko razlikuje stranke tako glede stikov kot odhodov.

Obe primerjavi računamo na **isti množici** strank z znanimi potrebnimi vrednostmi.
Neznanega podatka o podpori ne pretvorimo v »brez stikov«. Mera je agregirano
povprečje; iz nje ne rekonstruiramo dogodkov ali zadnjih 30 dni.


In [ ]:
primer_podpora = df[["churn", "custcare_Mean", "mou_Mean"]].dropna().copy()
primer_podpora["podpora"] = np.where(
    primer_podpora["custcare_Mean"].gt(0), "pozitivna mera stikov", "ničelna mera stikov"
)
primer_podpora["razred_uporabe"] = pd.qcut(
    primer_podpora["mou_Mean"], q=4, labels=["Q1: najmanj", "Q2", "Q3", "Q4: največ"]
)
print(f"Strank v obeh primerjavah: {len(primer_podpora):,}")
print("Surova primerjava:")
display(primer_podpora.groupby("podpora")["churn"].agg(
    stranke="size", delez_odhodov="mean"
).round(3))
print("Primerjava znotraj razredov uporabe:")
display(primer_podpora.groupby(["razred_uporabe", "podpora"], observed=True)["churn"].agg(
    stranke="size", delez_odhodov="mean"
).round(3))


**Razmisli:** ali razlika znotraj skupin ostane enaka? Ali to dokazuje, da klic podpore
prepreči odhod? Kaj še razlikuje skupini? Kontrola ene spremenljivke sama ne dokaže
vzročnosti. Potrebovali bi utemeljen vzročni načrt, na primer ustrezno zasnovan poskus.
